In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = {
    'Title': [
        'Інтерстеллар',
        'Марсіанин',
        'Гравітація',
        'Форсаж 9',
        'Місія нездійсненна',
        'Аватар',
        'Термінатор',
        'Пасажири'
    ],
    'Description': [
        'Подорож у космосі, час, місія врятувати людство',
        'Астронавт виживає на Марсі після аварії',
        'Астронавти борються за життя після катастрофи в космосі',
        'Гонки, автомобілі, швидкість, сімейні цінності',
        'Шпигунські місії, екшн, порятунок світу',
        'Інопланетяни, природа, колонізація, конфлікт',
        'Машини проти людей, майбутнє, битва за виживання',
        'Космічний корабель, любов, прокидання, експедиція'
    ]
}

df = pd.DataFrame(movies)

print("Список фільмів:")
print(df)

Список фільмів:
                Title                                        Description
0        Інтерстеллар    Подорож у космосі, час, місія врятувати людство
1           Марсіанин            Астронавт виживає на Марсі після аварії
2          Гравітація  Астронавти борються за життя після катастрофи ...
3            Форсаж 9     Гонки, автомобілі, швидкість, сімейні цінності
4  Місія нездійсненна            Шпигунські місії, екшн, порятунок світу
5              Аватар       Інопланетяни, природа, колонізація, конфлікт
6          Термінатор   Машини проти людей, майбутнє, битва за виживання
7            Пасажири  Космічний корабель, любов, прокидання, експедиція


In [8]:
vectorizer = TfidfVectorizer(stop_words=None)
tfidf_matrix = vectorizer.fit_transform(df['Description'])

similarity = cosine_similarity(tfidf_matrix)

def recommend_movie(title, n=3):
    if title not in df['Title'].values:
        return "Такого фільму немає в базі."
    
    idx = df[df['Title'] == title].index[0]
    
    sim_scores = list(enumerate(similarity[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    
    return df.iloc[movie_indices][['Title', 'Description']]

print("\nРекомендації для фільму 'Інтерстеллар':")
print(recommend_movie('Інтерстеллар'))


Рекомендації для фільму 'Інтерстеллар':
        Title                                        Description
2  Гравітація  Астронавти борються за життя після катастрофи ...
1   Марсіанин            Астронавт виживає на Марсі після аварії
3    Форсаж 9     Гонки, автомобілі, швидкість, сімейні цінності


In [12]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

ratings_data = {
    'User': ['Олег', 'Марина', 'Денис', 'Ірина', 'Сергій'],
    'Інтерстеллар': [5, 4, 0, 0, 5],
    'Марсіанин': [4, 5, 0, 0, 4],
    'Форсаж 9': [1, 1, 5, 4, 2],
    'Гравітація': [4, 5, 0, 3, 4],
    'Аватар': [3, 4, 5, 0, 5]
}

df = pd.DataFrame(ratings_data)
df.set_index('User', inplace=True)

print("Рейтинги користувачів:")
print(df)

Рейтинги користувачів:
        Інтерстеллар  Марсіанин  Форсаж 9  Гравітація  Аватар
User                                                         
Олег               5          4         1           4       3
Марина             4          5         1           5       4
Денис              0          0         5           0       5
Ірина              0          0         4           3       0
Сергій             5          4         2           4       5


In [13]:
user_similarity = pd.DataFrame(
    cosine_similarity(df),
    index=df.index,
    columns=df.index
)

print("\nСхожість між користувачами:")
print(user_similarity.round(2))


Схожість між користувачами:
User    Олег  Марина  Денис  Ірина  Сергій
User                                      
Олег    1.00    0.98   0.35   0.39    0.97
Марина  0.98    1.00   0.39   0.42    0.97
Денис   0.35    0.39   1.00   0.57    0.53
Ірина   0.39    0.42   0.57   1.00    0.43
Сергій  0.97    0.97   0.53   0.43    1.00


In [14]:
def recommend_for_user(username, n=3):
    if username not in df.index:
        return "Такого користувача немає."

    similar_users = user_similarity[username].sort_values(ascending=False)
    similar_users = similar_users[1:] 
    
    weighted_scores = pd.Series(0, index=df.columns, dtype=float)
    similarity_sum = pd.Series(0, index=df.columns, dtype=float)
    
    for user, sim in similar_users.items():
        for movie in df.columns:
            if df.loc[user, movie] > 0:
                weighted_scores[movie] += sim * df.loc[user, movie]
                similarity_sum[movie] += sim
    
    predicted_ratings = weighted_scores / similarity_sum
    predicted_ratings = predicted_ratings.fillna(0)

    unseen = df.loc[username] == 0
    recommendations = predicted_ratings[unseen].sort_values(ascending=False)
    
    return recommendations.head(n).round(2)

print("\nРекомендації для Дениса:")
print(recommend_for_user('Денис'))

print("\nРекомендації для Ірини:")
print(recommend_for_user('Ірина'))


Рекомендації для Дениса:
Інтерстеллар    4.69
Марсіанин       4.31
Гравітація      3.90
dtype: float64

Рекомендації для Ірини:
Інтерстеллар    4.66
Марсіанин       4.34
Аватар          4.34
dtype: float64
